In [ ]:
# Setup kann ignoriert werden

import base64
import functools
import json
import os
from datetime import date
from enum import Enum
from io import BytesIO
from pathlib import Path

import pypdfium2 as pdfium
from dotenv import load_dotenv
from IPython.display import Image, display
from openai import OpenAI
from pydantic import BaseModel, Field

# Vom Arbeitsverzeichnis aufwärts bis zum Repo-Root (dem Ordner mit .git)
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists())
DATA_DIR = REPO_ROOT / "backend" / "notebooks" / "data"
REQUEST_DIR = DATA_DIR / "request"
GENERATED_DIR = DATA_DIR / "generated"

# USD pro 1 Mio. Tokens. Vor einer echten Budgetplanung auf https://openai.com/api/pricing/ prüfen.
PRICES = {
    "gpt-6-luna": {"input": 0.10, "cached_input": 0.01, "output": 0.50},
    "gpt-4o-mini": {"input": 0.15, "cached_input": 0.075, "output": 0.60},
}


def usage_cost(usages, model: str) -> dict:
    """Tokens und Kosten (USD) eines usage-Objekts der Responses API oder einer Liste davon."""
    if not isinstance(usages, (list, tuple)):
        usages = [usages]
    input_tokens = cached_tokens = output_tokens = reasoning_tokens = 0
    for u in usages:
        input_tokens += u.input_tokens
        output_tokens += u.output_tokens
        cached_tokens += getattr(u.input_tokens_details, "cached_tokens", 0) or 0
        reasoning_tokens += getattr(u.output_tokens_details, "reasoning_tokens", 0) or 0
    price = PRICES[model]
    cost = ((input_tokens - cached_tokens) * price["input"]
            + cached_tokens * price["cached_input"]
            + output_tokens * price["output"]) / 1e6
    return {
        "input_tokens": input_tokens,
        "cached_tokens": cached_tokens,
        "output_tokens": output_tokens,  # enthält die Reasoning-Tokens
        "reasoning_tokens": reasoning_tokens,
        "cost_usd": cost,
    }


def print_cost(usages, model: str) -> None:
    c = usage_cost(usages, model)
    print(f"Input-Tokens:  {c['input_tokens']} (davon gecacht: {c['cached_tokens']})")
    print(f"Output-Tokens: {c['output_tokens']} (davon Reasoning: {c['reasoning_tokens']})")
    print(f"Geschätzte Kosten: ${c['cost_usd']:.6f}")


@functools.cache
def get_client() -> OpenAI:
    """Der OpenAI-Client, beim ersten Aufruf erstellt und danach wiederverwendet.

    Liest LLM_API_KEY und LLM_BASE_URL aus <repo>/.env.local, dieselben Variablen wie das Backend.
    Ein leeres LLM_BASE_URL bedeutet OpenAI selbst.
    """
    load_dotenv(REPO_ROOT / ".env.local")
    return OpenAI(api_key=os.environ["LLM_API_KEY"], base_url=os.getenv("LLM_BASE_URL") or None)


def image_to_data_url(path: Path) -> str:
    """Eine PNG-/JPEG-Datei als base64-Data-URL, für die `input_image`-Teile eines Responses-API-Aufrufs."""
    path = Path(path)
    mime = "image/jpeg" if path.suffix.lower() in {".jpg", ".jpeg"} else "image/png"
    return f"data:{mime};base64," + base64.b64encode(path.read_bytes()).decode()


def pdf_to_images(path: Path, dpi: int = 150) -> list[bytes]:
    """Jede Seite eines PDFs als PNG-Bytes (150 dpi reichen, um Kleingedrucktes zu lesen)."""
    pdf = pdfium.PdfDocument(path)
    images = []
    for page in pdf:
        buffer = BytesIO()
        page.render(scale=dpi / 72).to_pil().save(buffer, format="PNG")  # PDF-Punkte sind 1/72 Zoll
        images.append(buffer.getvalue())
    return images


def pdf_to_data_urls(path: Path, dpi: int = 150) -> list[str]:
    """Eine Data-URL pro Seite, für die `input_image`-Teile eines Responses-API-Aufrufs."""
    return ["data:image/png;base64," + base64.b64encode(png).decode() for png in pdf_to_images(path, dpi)]


def save_pdf_pages(path: Path, out_dir: Path = GENERATED_DIR, dpi: int = 150) -> list[Path]:
    """Speichert jede Seite als <PDF-Name>_page1.png, _page2.png, ... und gibt die Pfade zurück (z. B. zum Anzeigen)."""
    path = Path(path)
    out_dir.mkdir(parents=True, exist_ok=True)
    paths = []
    for number, png in enumerate(pdf_to_images(path, dpi), start=1):
        paths.append(out_dir / f"{path.stem}_page{number}.png")
        paths[-1].write_bytes(png)
    return paths


client = get_client()


# Optionale Übungen

Wenden Sie nun dieselben Schritte auf die fiktionale Anfrage in `data/request/` an. **Das `SupplierRequest`-Modell aus Übung 2 ist das Schema, das für den Rest des Tages verwendet wird.**

In der nächsten Zelle wählen Sie das Modell für alle Aufrufe in diesem Notebook: `gpt-6-luna` (aktuell, Standard) oder `gpt-4o-mini` (älter). Probieren Sie beide aus und vergleichen Sie die Ergebnisse.

In [ ]:
# Modell für alle Aufrufe in diesem Notebook: "gpt-6-luna" oder "gpt-4o-mini"
MODEL = "gpt-6-luna"

In [ ]:
REQUEST_PDF = REQUEST_DIR / "Anfrage_ANF-2026-0418_Hegibach.pdf"

for page in save_pdf_pages(REQUEST_PDF):
    display(Image(filename=page, width=800))


### Übung 1: Ein JSON-Schema für die Anfrage

Schreiben Sie ein striktes JSON-Schema für die Anfrage mit diesen Feldern:
- `contact_name` als string
- `customer_company` als string
- `quote_deadline` als Datum im ISO-8601 Format; YYYY-MM-DD.,
- `requested_delivery_date` als Datum im ISO-8601 Format; YYYY-MM-DD.
- `currency`, als enum
- `certificates_required`, als array mit strings

und extrahieren Sie damit Anfrage_ANF-2026-0418_Hegibach.

Ein Beispiel für arrays mit integer:

````
"Number of items per line item": {
            "type": "array",
            "items": {"type": "integer"},
            "description": "Fill in how many times each line item has been requested.",
        }
````

In [ ]:
# TODO (Übung 1): Schreiben Sie das JSON-Schema für die Anfrage, mit einem Enum für die Währung
request_schema = {
    "type": "object",
    "properties": {
        # TODO: Hier ergänzen!
    }
}

In [ ]:
# Lösung
request_schema = {
    "type": "object",
    "properties": {
        "contact_name": {"type": "string", "description": "The buyer who sent the request, as: Firstname Lastname."},
        "customer_company": {"type": "string", "description": "The company that sends the request and will buy the parts."},
        "quote_deadline": {"type": "string", "format": "date", "description": "Date by which the quote must arrive. ISO-8601 formatted date; YYYY-MM-DD."},
        "requested_delivery_date": {"type": "string", "format": "date", "description": "Delivery date for the first batch, ISO-8601 formatted date; YYYY-MM-DD."},
        "requested_delivery_string": {"type": "string", "description": "Delivery date for the first batch, as written in the request."},
        "currency": {"type": "string", "enum": ["CHF", "EUR", "USD"], "description": "Currency in which the quote is to be made."},
        "certificates_required": {
            "type": "array",
            "items": {"type": "string"},
            "description": "Every certificate or test report the supplier must provide, with the positions it applies to.",
        },
    },
    "required": ["requested_delivery_string", "contact_name", "customer_company", "quote_deadline", "requested_delivery_date", "currency", "certificates_required"],
    "additionalProperties": False,
}

In [ ]:
# Testen Sie, ob es funktioniert!
page_images = [{"type": "input_image", "image_url": url} for url in pdf_to_data_urls(REQUEST_PDF)]
response = client.responses.create(
    model=MODEL,
    input=[{"role": "user", "content": [{"type": "input_text", "text": "Extract the request fields."}, *page_images]}],
    text={"format": {"type": "json_schema", "name": "SupplierRequest", "schema": request_schema, "strict": True}},
)
json.loads(response.output_text)

### Übung 2: Die Anfrage als Pydantic-Modell

Schreiben Sie `SupplierRequest` als Pydantic-Modell mit denselben Feldern wie das JSON-Schema aus Übung 1:
- `contact_name` als `str`
- `customer_company` als `str`
- `quote_deadline` als `date`
- `requested_delivery_date` als `date`
- `currency` als Enum
- `certificates_required` als `list[str]`

Extrahieren Sie wieder Anfrage_ANF-2026-0418_Hegibach, diesmal mit `client.responses.parse`.

In [ ]:
# TODO (Übung 2): Schreiben Sie SupplierRequest als Pydantic-Modell
class Currency(str, Enum):
    # TODO: Hier ergänzen!
    pass


class SupplierRequest(BaseModel):
    # TODO: Hier ergänzen!
    pass

In [ ]:
# LÖSUNGEN
class Currency(str, Enum):
    CHF = "CHF"
    EUR = "EUR"
    USD = "USD"


class SupplierRequest(BaseModel):
    contact_name: str = Field(description="The buyer who sent the request, as: Firstname Lastname.")
    customer_company: str = Field(description="The company that sends the request and will buy the parts.")
    quote_deadline: date = Field(description="Date by which the quote must arrive. ISO-8601 formatted date; YYYY-MM-DD.")
    requested_delivery_date: date = Field(description="Delivery date for the first batch, ISO-8601 formatted date; YYYY-MM-DD.")
    requested_delivery_string: str = Field(description="Delivery date for the first batch, as written in the request.")
    currency: Currency = Field(description="Currency in which the quote is to be made.")
    certificates_required: list[str] = Field(
        description="Every certificate or test report the supplier must provide, with the positions it applies to."
    )

In [ ]:
# Testen Sie die Lösung!
page_images = [{"type": "input_image", "image_url": url} for url in pdf_to_data_urls(REQUEST_PDF)]
response = client.responses.parse(
    model=MODEL,
    input=[{"role": "user", "content": [{"type": "input_text", "text": "Extract the request fields into the provided schema."}, *page_images]}],
    text_format=SupplierRequest,
)
request = response.output_parsed
request

### Übung 3: Verschachtelte Modelle

Bisher bestand die Anfrage aus einfachen Feldern. Eine echte Anfrage enthält mehr: eine Tabelle mit Positionen und Informationen, die *fehlen*.

Viele Anfragen haben fehlende Informationen, deswegen kreieren wir noch ein weiteres Feld, um diese möglichst abzufangen. Zusätzlich möchte man oftmals mehrere Listen von Informationen miteinander verbinden. z.B. hat man 10 Personen. Alle haben einen Vor- und Nachnamen, Alter und Geschlecht. Anstatt 4 Listen zu kreieren für jedes Attribut, machen wir eine einzelne Liste mit einer Personen, wo jede Person mehrere Attribute besitzen kann.


Die Hilfsfunktion unten schickt beide Seiten der Anfrage zusammen mit einem Prompt, so wie `image_input` im Explainer-Notebook es für ein einzelnes Bild tut.

In [ ]:
def request_input(prompt: str, path: Path = REQUEST_PDF) -> list[dict]:
    """Eine User-Nachricht mit unserem Prompt und einem Bild pro Seite des PDFs."""
    pages = [{"type": "input_image", "image_url": url} for url in pdf_to_data_urls(path)]
    return [{"role": "user", "content": [{"type": "input_text", "text": prompt}, *pages]}]

#### Übung 3.1: Fehlende Informationen

Für ein verbindliches Angebot braucht der Lieferant alle Angaben. In der Anfrage fehlt manches oder ist noch offen. Erweitern Sie `SupplierRequest` um das Feld `missing_information` als `list[str]`: ein kurzer Eintrag pro fehlender Information.

Mit Vererbung geht das kurz: `class RequestWithMissing(SupplierRequest)` übernimmt alle Felder und fügt eines hinzu. Die `description` ist hier entscheidend, denn sie sagt dem Modell, was als «fehlend» gilt.

Lesen Sie danach die Anfrage selbst nochmals durch: Hat das Modell alles gefunden (Positionen, Beilagen, Hinweise)? Passen Sie sonst die `description` an und führen Sie die Zelle nochmals aus.

In [ ]:
# TODO (Übung 3.1): Ergänzen Sie die Anfrage um missing_information

class RequestWithMissing(BaseModel):
    contact_name: str = Field(description="The buyer who sent the request, as: Firstname Lastname.")
    customer_company: str = Field(description="The company that sends the request and will buy the parts.")
    quote_deadline: date = Field(description="Date by which the quote must arrive. ISO-8601 formatted date; YYYY-MM-DD.")
    requested_delivery_date: date = Field(description="Delivery date for the first batch, ISO-8601 formatted date; YYYY-MM-DD.")
    requested_delivery_string: str = Field(description="Delivery date for the first batch, as written in the request.")
    currency: Currency = Field(description="Currency in which the quote is to be made.")
    certificates_required: list[str] = Field(
        description="Every certificate or test report the supplier must provide, with the positions it applies to."
    )
    # TODO: Fügen sie noch das Attribut für fehlende Informationen hier dazu

# Kann auch folgendermassen gelöst werden:
#  
# class RequestWithMissing(SupplierRequest):
#     # TODO: Hier ergänzen!
#     pass

In [ ]:
# LÖSUNGEN
class RequestWithMissing(SupplierRequest):
    missing_information: list[str] = Field(
        description="Information needed to calculate a binding quote that is missing or still open. One short item each."
    )

# Tipp: Robuster ist es, im Code zu prüfen, welche Felder null sind, statt das Modell erraten zu lassen, was fehlt (siehe Teil 5).

In [ ]:
# Testen Sie es!
response = client.responses.parse(
    model=MODEL,
    input=request_input("Extract the request fields into the provided schema."),
    text_format=RequestWithMissing,
)
for item in response.output_parsed.missing_information:
    print("-", item)

#### Übung 3.2: Positionen als eigenes Modell

Die Tabelle «Positionen» ist eine Liste gleichartiger Objekte. Dafür schreiben wir ein eigenes, reduziertes Modell `LineItem` mit vier Feldern:
- `position` als `int`
- `description` als `str`
- `material` als `str`, oder `None`, wenn es nicht angegeben ist
- `quantities` als `list[int]`: alle angefragten Mengenstaffeln, die kleinste zuerst

Danach fügen Sie es als `line_items: list[LineItem]` in die Anfrage ein. Ein Pydantic-Modell kann als Typ eines Feldes in einem anderen Modell stehen, beliebig tief. So entstehen aus einfachen Bausteinen mächtige Modelle: Die API liefert für jede Position ein eigenes Objekt mit den richtigen Typen.

In einer echten Anwendung hätte `LineItem` weitere Felder (Zeichnungsnummer, Prozesse, Oberfläche, ...). Hier bleibt es bewusst klein.

In [ ]:
# TODO (Übung 3.2): Schreiben Sie LineItem und ergänzen Sie die Anfrage um line_items
class LineItem(BaseModel):
    # TODO: Hier ergänzen!
    pass

class RequestWithLineItems(BaseModel):
    contact_name: str = Field(description="The buyer who sent the request, as: Firstname Lastname.")
    customer_company: str = Field(description="The company that sends the request and will buy the parts.")
    quote_deadline: date = Field(description="Date by which the quote must arrive. ISO-8601 formatted date; YYYY-MM-DD.")
    requested_delivery_date: date = Field(description="Delivery date for the first batch, ISO-8601 formatted date; YYYY-MM-DD.")
    requested_delivery_string: str = Field(description="Delivery date for the first batch, as written in the request.")
    currency: Currency = Field(description="Currency in which the quote is to be made.")
    certificates_required: list[str] = Field(
        description="Every certificate or test report the supplier must provide, with the positions it applies to."
    )
    missing_information: list[str] = Field(
        description="Information needed to calculate a binding quote that is missing or still open. One short item each."
    )
    # TODO: Hier wollen wir eine Liste von LineItem

#  Kann auch folgendermassen gelöst werden:
# 
# class RequestWithLineItems(RequestWithMissing):
#     # TODO: Hier ergänzen!
#     pass

In [ ]:
# LÖSUNGEN
class LineItem(BaseModel):
    position: int = Field(description="Position number as written in the request, e.g. 10, 20.")
    description: str
    material: str | None = Field(description="Material designation exactly as written. Null if not specified.")
    quantities: list[int] = Field(description="All requested quantity tiers, smallest first.")


class RequestWithLineItems(RequestWithMissing):
    line_items: list[LineItem]

In [ ]:
# Testen Sie es!
response = client.responses.parse(
    model=MODEL,
    input=request_input("Extract the request fields into the provided schema."),
    text_format=RequestWithLineItems,
)
request = response.output_parsed
for item in request.line_items:  # LineItem-Objekte, keine Dicts
    print(item.position, item.description, item.material, item.quantities)

In [ ]:
# Das verschachtelte Modell steht einmal unter $defs, das Feld verweist mit $ref darauf
schema = RequestWithLineItems.model_json_schema()
print("$defs:", list(schema["$defs"]))
print("line_items:", schema["properties"]["line_items"])

# Verschachtelte Werte sind typisierte Python-Objekte, damit kann man direkt rechnen
print("Stück in der ersten Lieferung:", sum(item.quantities[0] for item in request.line_items))

### 4: Extraktionsmethoden vergleichen

Dieser Teil ist bereits vorbereitet und baut auf Ihren bisherigen Lösungen auf. Stellen Sie also sicher, dass Sie die Übungen 1 und 2 abgeschlossen haben, bevor Sie ihn ausführen. Er macht Folgendes:

1. Er extrahiert die Anfrage mit beiden Methoden: dem JSON-Schema (Übung 1, `request_schema`) und Pydantic (Übung 2, `SupplierRequest`). Beide haben dieselben Felder.
2. Er führt jede Methode 3 Mal aus. Variieren die Ergebnisse von Durchlauf zu Durchlauf?
3. Er vergleicht beide mit der Ground Truth (`TRUTH`, von Hand geprüft) und gibt jedes Feld aus, das davon abweicht. Datum und Freitext lassen sich nicht mit `==` vergleichen: Beim Lieferdatum («KW 2/2027») muss nur die Kalenderwoche stimmen, und bei den Zertifikaten muss das Prüfzeugnis 3.1 vorkommen.
4. Er berechnet die Gesamtkosten aller Aufrufe.

Spielen Sie ein wenig damit herum: Setzen Sie `MODEL` oben auf `gpt-4o-mini` bzw. `gpt-6-luna` und schauen Sie, ob sich die Ergebnisse unterscheiden.

In [ ]:
# Verhindert, dass dieser Teil mit «Run All» versehentlich mitläuft: Er dauert etwa 20 Sekunden und ruft das LLM mehrmals auf.
raise RuntimeError("Führen Sie die nächste Zelle manuell aus!")

In [ ]:
# Die von Hand geprüfte Lösung dieser Anfrage
TRUTH = {
    "contact_name": "Sandra Keller",
    "customer_company": "Hegibach Anlagenbau AG",
    "quote_deadline": "2026-11-06",
    "delivery_week": (2027, 2),  # "KW 2/2027"
    "currency": "CHF",
    "certificate": "3.1",  # das Abnahmeprüfzeugnis EN 10204 3.1 muss unter den Zertifikaten sein
}

RUNS = 3

def iso_week(value) -> tuple[int, int]:
    """(Jahr, Woche) eines Datums oder eines ISO-8601-Datumsstrings."""
    return date.fromisoformat(str(value)).isocalendar()[:2]


# Eine Prüfung pro Feld: Datum und Freitext brauchen einen lockereren Vergleich als ==
CHECKS = {
    "contact_name": lambda v: v == TRUTH["contact_name"],
    "customer_company": lambda v: v == TRUTH["customer_company"],
    "quote_deadline": lambda v: str(v) == TRUTH["quote_deadline"],
    # "KW 2/2027" ist kein einzelnes Datum: Nur die Kalenderwoche muss stimmen
    "requested_delivery_date": lambda v: iso_week(v) == TRUTH["delivery_week"],
    "currency": lambda v: v == TRUTH["currency"],
    "certificates_required": lambda v: any(TRUTH["certificate"] in certificate for certificate in v),
}


def matches(field, value) -> bool:
    try:
        return CHECKS[field](value)
    except (ValueError, TypeError):  # z. B. ein Datumsstring, der kein Datum ist
        return False


usages = []  # die usage jedes Aufrufs, für die Gesamtkosten

def extract_with_schema():
    response = client.responses.create(
        model=MODEL,
        input=request_input("Extract the request fields."),
        text={"format": {"type": "json_schema", "name": "SupplierRequest", "schema": request_schema, "strict": True}},
    )
    usages.append(response.usage)
    return json.loads(response.output_text)

def extract_with_pydantic():
    response = client.responses.parse(
        model=MODEL,
        input=request_input("Extract the request fields."),
        text_format=SupplierRequest,
    )
    usages.append(response.usage)
    return response.output_parsed.model_dump()


for run in range(RUNS):
    extracted = {"schema": extract_with_schema(), "pydantic": extract_with_pydantic()}
    correct = {"schema": 0, "pydantic": 0}
    deviations = []  # jedes Feld, das in diesem Durchlauf von der Ground Truth abweicht
    for method, values in extracted.items():
        for f in CHECKS:
            if matches(f, values.get(f)):
                correct[method] += 1
            else:
                deviations.append(f"  {method:<8} {f}: erhalten {values.get(f)!r}")
    print(f"Durchlauf {run}: korrekte Felder, Schema {correct['schema']}/{len(CHECKS)}, Pydantic {correct['pydantic']}/{len(CHECKS)}")
    print("\n".join(deviations) if deviations else "  keine Abweichungen")

print(f"\nTotal über {len(usages)} Aufrufe:")
print_cost(usages, MODEL)

### 5: Ein vollständiges Beispiel

Zum Schluss ein Modell, das sich für beliebige Anfragen (RFQs) eignen kann, nicht nur für diese eine. Es kombiniert alle Bausteine aus diesem Notebook und folgt einigen Regeln, die sich bei den Versuchen oben bewährt haben:

- **Beide Parteien**: `customer` (wer anfragt und kauft) und `supplier` (wer das Angebot schreibt) sind getrennt. So verwechselt das Modell die Einkäuferin nicht mit dem Empfänger der Anfrage. Kontaktpersonen sind eine Liste, ihre Funktion steht in `role`.
- **Wie geschrieben vs. normalisiert**: Werkstoff und Liefertermin gibt es zweimal, zum Beispiel `material_as_written` («Aluminium, Legierung nach Ihrer Empfehlung») und `material` (`null`, weil die Legierung offen ist). Das Modell muss nicht zwischen Abschreiben und Interpretieren wählen.
- **Klare Sprachregel**: Nummern, Werkstoffe und Bezeichnungen genau wie geschrieben; Freitext in der Sprache des Dokuments.
- **Enums mit Ausweg**: `Process` hat den Wert `other`, die Details stehen in `process_notes`. Eine Anfrage mit Fräs- oder Gussteilen passt so trotzdem ins Schema.
- **Mengen mit Einheit und Bedeutung**: Jede Mengenstaffel hat eine Menge, eine Einheit, eine Bezeichnung (Nullserie, Jahresbedarf, ...) und die Anzahl Abrufe.
- **Anforderungen mit Geltungsbereich**: Zertifikate, Toleranzen, Oberflächen, Verpackung usw. sind je ein `Requirement` mit `applies_to_positions` (`null` = alle Positionen).
- **Beilagen statt Ja/Nein-Felder**: Das Modell listet nur die Beilagen auf. Ob eine Zeichnung oder STEP-Datei fehlt, berechnet der Code. Das ist deterministisch und übersieht nichts.
- **Kaufmännische Bedingungen gebündelt**: Währung, Zahlungsfrist, Angebotsbindung und Incoterm (als Enum, mit Ort und Version) stehen in `CommercialTerms`. Es gibt keine Standardwerte: Was nicht im Dokument steht, ist `null`.
- **Auffangfeld**: Was in kein Feld passt, landet in `notes`, statt verloren zu gehen.

**Für eine andere Anfrage:** Setzen Sie `RFQ_PATH` auf die neue Datei (PDF oder PNG) und führen Sie die Zellen aus. Passen Sie bei Bedarf `REQUIRED_FILE_TYPES` an. Nur der Vergleich am Ende braucht eine Lösung (`EXPECTED`) für genau diese Anfrage.

In [ ]:
from typing import Literal

class Contact(BaseModel):
    name: str
    role: str | None = Field(None, description="Funktion oder Abteilung, z. B. Einkauf, Konstruktion.")
    email: str | None = None
    phone: str | None = None


class Party(BaseModel):
    company: str = Field(description="Firmenname in normaler Schreibweise, nicht in Grossbuchstaben aus einem Briefkopf.")
    address: str | None = Field(None, description="Adresse in einer Zeile, wie geschrieben.")
    contacts: list[Contact] = Field(description="Alle genannten Kontaktpersonen dieser Firma.")


class Process(str, Enum):
    laser_cutting = "laser_cutting"
    punching = "punching"
    sawing = "sawing"
    bending = "bending"
    welding = "welding"
    turning = "turning"
    milling = "milling"
    drilling = "drilling"
    threading = "threading"
    insertion_of_fasteners = "insertion_of_fasteners"
    assembly = "assembly"
    other = "other"


class QuantityTier(BaseModel):
    quantity: float
    unit: str = Field(description="Einheit wie geschrieben, z. B. Stk., kg, m.")
    label: str | None = Field(None, description="Bedeutung der Staffel, z. B. Nullserie, Jahresbedarf, Prototyp.")
    call_offs: int | None = Field(None, description="Anzahl Abrufe oder Lose, falls angegeben.")


class LineItem(BaseModel):
    position: str = Field(description="Positionsnummer wie geschrieben, z. B. 10, 1.2.")
    description: str = Field(description="Bezeichnung genau wie in der Positionsliste, ohne zusätzliche Hinweise.")
    drawing_number: str | None = Field(None, description="Zeichnungsnummer ohne Revision.")
    drawing_revision: str | None = None
    material_as_written: str | None = Field(None, description="Werkstoff genau wie geschrieben.")
    material: str | None = Field(
        None,
        description="Werkstoffbezeichnung nach Norm (z. B. S235JR, 1.4301, EN AW-5754). "
        "Null, wenn der Kunde den Werkstoff ganz oder teilweise offen lässt.",
    )
    dimensions_as_written: str | None = Field(
        None, description="Dicke, Durchmesser oder Abmessungen wie geschrieben, z. B. '3 mm'. Null, wenn nicht angegeben."
    )
    processes: list[Process] = Field(description="Alle Bearbeitungsschritte dieser Position.")
    process_notes: str | None = Field(None, description="Nur wenn processes 'other' enthält: welche Bearbeitung gemeint ist. Sonst null.")
    surface_treatment: str | None = Field(None, description="Oberfläche genau wie geschrieben.")
    quantities: list[QuantityTier] = Field(description="Alle Mengenstaffeln, die kleinste zuerst.")


class Requirement(BaseModel):
    kind: Literal["certificate", "tolerance", "surface", "packaging", "quality", "other"]
    text: str = Field(description="Die Anforderung, kurz und in der Sprache des Dokuments.")
    applies_to_positions: list[str] | None = Field(
        None, description="Positionsnummern, für die sie gilt. Null, wenn sie für alle Positionen gilt."
    )


class Attachment(BaseModel):
    filename: str = Field(description="Dateiname genau wie in der Liste der Beilagen.")
    file_type: str = Field(description="Dateiendung in Kleinbuchstaben, z. B. pdf, step, dxf.")
    drawing_number: str | None = Field(None, description="Zeichnungsnummer, zu der die Datei gehört, ohne Revision.")


class Incoterm(str, Enum):
    EXW = "EXW"
    FCA = "FCA"
    CPT = "CPT"
    CIP = "CIP"
    DAP = "DAP"
    DPU = "DPU"
    DDP = "DDP"
    FAS = "FAS"
    FOB = "FOB"
    CFR = "CFR"
    CIF = "CIF"


class CommercialTerms(BaseModel):
    currency: str | None = Field(None, description="ISO-4217-Code, z. B. CHF, EUR. Null, wenn nicht angegeben.")
    payment_terms_days: int | None = Field(None, description="Zahlungsfrist in Tagen.")
    quote_validity_days: int | None = Field(None, description="Wie lange das Angebot mindestens gelten muss, in Tagen.")
    incoterm: Incoterm | None = None
    incoterm_place: str | None = Field(None, description="Ort zum Incoterm, z. B. Winterthur.")
    incoterm_version: str | None = Field(None, description="Version der Incoterms, z. B. 2020.")
    one_time_costs_separate: bool | None = Field(
        None, description="True, wenn einmalige Kosten (Programmierung, Vorrichtungen) separat auszuweisen sind. Null, wenn nicht erwähnt."
    )


class Delivery(BaseModel):
    as_written: str | None = Field(None, description="Gewünschter Liefertermin genau wie geschrieben, z. B. KW 2/2027.")
    iso_week: str | None = Field(None, description="Derselbe Termin als ISO-Woche im Format YYYY-Www, z. B. 2027-W02.")
    exact_date: date | None = Field(None, description="Nur wenn ein genaues Datum genannt ist. ISO 8601: YYYY-MM-DD.")


class OpenPoint(BaseModel):
    position: str | None = Field(None, description="Positionsnummer, oder null, wenn der Punkt die ganze Anfrage betrifft.")
    text: str

# Das ist ein generelles Modell für eine Offerte, so wie wir sie uns vorstellen. Überprüfen Sie die Einträge und schauen Sie inwiefern es auf Ihre Firma zutrifft.
# Was würden Sie ändern und was würden Sie beibehalten?

# Arbeiten sie von dieser Klasse Rückwärts
class RFQ(BaseModel):
    rfq_number: str | None = None
    rfq_date: date | None = Field(None, description="Datum der Anfrage. ISO 8601: YYYY-MM-DD.")
    project_name: str | None = None
    customer: Party = Field(description="Die Firma, die anfragt und kaufen will.")
    supplier: Party = Field(description="Die Firma, an die die Anfrage gerichtet ist und die das Angebot schreibt.") #Wenn das immer ihre Firma ist, können Sie das löschen.
    supplier_number: str | None = Field(None, description="Lieferantennummer des Lieferanten beim Kunden.")
    quote_deadline: date | None = Field(None, description="Bis wann das Angebot eintreffen muss. ISO 8601: YYYY-MM-DD.")
    delivery: Delivery
    terms: CommercialTerms
    requirements: list[Requirement] = Field(description="Alle technischen und kaufmännischen Anforderungen an das Angebot oder die Teile.")
    line_items: list[LineItem]
    attachments: list[Attachment] = Field(description="Alle Dateien, die die Anfrage als Beilage nennt.")
    open_points: list[OpenPoint] = Field(
        description="Punkte, die der Text ausdrücklich als offen oder nachgereicht nennt. "
        "Leere Felder oben hier nicht wiederholen."
    )
    notes: list[str] = Field(description="Alles Weitere, das für das Angebot wichtig ist und in kein anderes Feld passt.")

In [ ]:
# Diesen System Prompt kann man auch nach Belieben tunen.
SYSTEM_PROMPT = """Du extrahierst die Angaben aus einer Anfrage (RFQ) an einen Fertigungsbetrieb.
- Nummern, Zeichnungen, Werkstoffe und Bezeichnungen genau so übernehmen, wie sie im Dokument stehen.
- Freitext kurz und in der Sprache des Dokuments.
- Was nicht im Dokument steht, ist null. Nicht raten."""

# Welche Dateien zu jeder Zeichnung beiliegen müssen, damit wir kalkulieren können
REQUIRED_FILE_TYPES = {"pdf", "step"}


def extract_rfq(path: Path, model: str | None = None):
    """Liest eine Anfrage (PDF oder PNG). Gibt (RFQ, usage) zurück."""
    path = Path(path)
    model = model or MODEL
    urls = pdf_to_data_urls(path) if path.suffix.lower() == ".pdf" else [image_to_data_url(path)]
    response = client.responses.parse(
        model=model,
        input=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": [
                {"type": "input_text", "text": "Extrahiere die Anfrage."},
                *[{"type": "input_image", "image_url": url} for url in urls],
            ]},
        ],
        text_format=RFQ,
    )
    return response.output_parsed, response.usage


def missing_information(rfq: RFQ) -> list[str]:
    """Fehlende Angaben, im Code abgeleitet aus leeren Feldern und den Beilagen."""
    attached = {(a.drawing_number, a.file_type.lower()) for a in rfq.attachments}
    missing = []
    if rfq.quote_deadline is None:
        missing.append("Angebotsfrist fehlt")
    if rfq.delivery.as_written is None:
        missing.append("Liefertermin fehlt")
    if rfq.terms.currency is None:
        missing.append("Währung fehlt")
    for item in rfq.line_items:
        pos = f"Pos. {item.position}"
        if item.material is None:
            missing.append(f"{pos}: Werkstoff nicht festgelegt")
        if item.dimensions_as_written is None:
            missing.append(f"{pos}: Dicke bzw. Abmessung fehlt")
        if item.drawing_number is None:
            missing.append(f"{pos}: keine Zeichnungsnummer")
            continue
        for file_type in sorted(REQUIRED_FILE_TYPES):
            if (item.drawing_number, file_type) not in attached:
                missing.append(f"{pos}: {file_type.upper()}-Datei zu {item.drawing_number} fehlt in den Beilagen")
    return missing

In [ ]:
RFQ_PATH = REQUEST_DIR / "Anfrage_ANF-2026-0418_Hegibach.pdf"  # hier kann man für Testzwecke eine andere Anfrage einsetzen

rfq, usage = extract_rfq(RFQ_PATH)
print(rfq.model_dump_json(indent=2))

print("\nFehlende Angaben (im Code abgeleitet):")
for item in missing_information(rfq):
    print("-", item)
print("\nOffene Punkte (laut Anfrage):")
for point in rfq.open_points:
    print("-", f"Pos. {point.position}: {point.text}" if point.position else point.text)
print()
print_cost(usage, MODEL)

#### Vergleich mit der Lösung -- Das ist Testcode und muss nicht im Detail verstanden werden. Es dient nur als Test.

`EXPECTED` ist die von Hand geprüfte Lösung für Anfrage ANF-2026-0418. Der Vergleich ordnet Positionen, Beilagen und Kontakte über ihre Nummer, ihren Dateinamen bzw. ihren Namen zu, nicht über die Reihenfolge. Er unterscheidet drei Arten von Feldern:

- **Strukturierte Felder** (Nummern, Daten, Enums, Werkstoffe, Mengen) müssen exakt stimmen und werden gezählt.
- **Freitext** (`requirements`, `open_points`, `notes`, Adressen, Bezeichnungen der Mengenstaffeln) lässt sich nicht mit `==` vergleichen. Diese Felder werden nur zur Sichtprüfung angezeigt.
- **Fehlende Angaben** berechnet `missing_information` im Code, für die Extraktion und für `EXPECTED`. Weil der Code sie formuliert, lassen sie sich exakt vergleichen.

In [ ]:
# Die von Hand geprüfte Lösung: was eine perfekte Extraktion dieser Anfrage liefern sollte.
EXPECTED = RFQ(
    rfq_number="ANF-2026-0418",
    rfq_date=date(2026, 10, 19),
    project_name="Förderband-Modul FB-300",
    customer=Party(
        company="Hegibach Anlagenbau AG",
        address="Industriestrasse 14, 8404 Winterthur, Schweiz",
        contacts=[
            Contact(name="Sandra Keller", role="Einkauf",
                    email="s.keller@hegibach-anlagenbau.example", phone="+41 52 000 14 27"),
            Contact(name="Daniel Frei", role="Konstruktion",
                    email="d.frei@hegibach-anlagenbau.example", phone="+41 52 000 14 51"),
        ],
    ),
    supplier=Party(
        company="Tössthal Blechtechnik GmbH",
        address="Bahnhofstrasse 3, 8488 Turbenthal",
        contacts=[Contact(name="Marco Brunner", role="Verkauf Innendienst")],
    ),
    supplier_number="L-20733",
    quote_deadline=date(2026, 11, 6),
    delivery=Delivery(as_written="KW 2/2027", iso_week="2027-W02"),
    terms=CommercialTerms(
        currency="CHF", payment_terms_days=30, quote_validity_days=60,
        incoterm=Incoterm.DAP, incoterm_place="Winterthur", incoterm_version="2020",
        one_time_costs_separate=True,
    ),
    requirements=[
        Requirement(kind="tolerance", text="Allgemeintoleranzen nach ISO 2768-mK, sofern auf der Zeichnung nicht anders angegeben"),
        Requirement(kind="certificate", text="Abnahmeprüfzeugnis 3.1 nach EN 10204", applies_to_positions=["30", "50"]),
        Requirement(kind="quality", text="Biegerichtung zwingend einhalten; Kratzer auf der Sichtseite nicht zulässig",
                    applies_to_positions=["10", "20"]),
        Requirement(kind="surface", text="Sichtteil, Schweissnähte verschleifen", applies_to_positions=["40"]),
        Requirement(kind="packaging", text="In Gitterboxen, Sichtteile einzeln mit Zwischenlage"),
        Requirement(kind="quality", text="Qualitätsmanagementsystem nach ISO 9001 oder gleichwertig"),
        Requirement(kind="quality", text="Erstmusterprüfberichte auf Verlangen kostenlos beilegen"),
    ],
    line_items=[
        LineItem(position="10", description="Seitenblech links", drawing_number="HB-30021", drawing_revision="C",
                 material_as_written="S235JR", material="S235JR", dimensions_as_written="3 mm",
                 processes=[Process.laser_cutting, Process.bending], surface_treatment="pulverbeschichtet RAL 7035",
                 quantities=[QuantityTier(quantity=40, unit="Stk.", label="Nullserie"),
                             QuantityTier(quantity=120, unit="Stk.", label="Jahresbedarf Serie", call_offs=4)]),
        LineItem(position="20", description="Seitenblech rechts (spiegelbildlich zu Pos. 10)", drawing_number="HB-30022",
                 drawing_revision="C", material_as_written="S235JR", material="S235JR", dimensions_as_written="3 mm",
                 processes=[Process.laser_cutting, Process.bending], surface_treatment="pulverbeschichtet RAL 7035",
                 quantities=[QuantityTier(quantity=40, unit="Stk.", label="Nullserie"),
                             QuantityTier(quantity=120, unit="Stk.", label="Jahresbedarf Serie", call_offs=4)]),
        LineItem(position="30", description="Montagewinkel", drawing_number="HB-30110", drawing_revision="A",
                 material_as_written="1.4301", material="1.4301", dimensions_as_written="2 mm",
                 processes=[Process.laser_cutting, Process.bending, Process.insertion_of_fasteners],
                 surface_treatment="geschliffen K240",
                 quantities=[QuantityTier(quantity=240, unit="Stk.", label="Nullserie"),
                             QuantityTier(quantity=720, unit="Stk.", label="Jahresbedarf Serie", call_offs=4)]),
        LineItem(position="40", description="Abdeckhaube", drawing_number="HB-30200", drawing_revision="B",
                 material_as_written="Aluminium, Legierung nach Ihrer Empfehlung", material=None, dimensions_as_written=None,
                 processes=[Process.laser_cutting, Process.bending, Process.welding],
                 surface_treatment="eloxiert natur (E6/EV1)",
                 quantities=[QuantityTier(quantity=40, unit="Stk.", label="Nullserie"),
                             QuantityTier(quantity=120, unit="Stk.", label="Jahresbedarf Serie", call_offs=4)]),
        LineItem(position="50", description="Distanzbolzen M8 x 45", drawing_number="HB-30305", drawing_revision="A",
                 material_as_written="1.4305", material="1.4305", dimensions_as_written="Ø 16 mm",
                 processes=[Process.turning, Process.threading], surface_treatment="blank",
                 quantities=[QuantityTier(quantity=480, unit="Stk.", label="Nullserie"),
                             QuantityTier(quantity=1440, unit="Stk.", label="Jahresbedarf Serie", call_offs=4)]),
    ],
    attachments=[
        Attachment(filename="HB-30021_C.pdf", file_type="pdf", drawing_number="HB-30021"),
        Attachment(filename="HB-30021_C.step", file_type="step", drawing_number="HB-30021"),
        Attachment(filename="HB-30022_C.pdf", file_type="pdf", drawing_number="HB-30022"),
        Attachment(filename="HB-30022_C.step", file_type="step", drawing_number="HB-30022"),
        Attachment(filename="HB-30110_A.pdf", file_type="pdf", drawing_number="HB-30110"),
        Attachment(filename="HB-30110_A.step", file_type="step", drawing_number="HB-30110"),
        Attachment(filename="HB-30305_A.pdf", file_type="pdf", drawing_number="HB-30305"),
    ],
    open_points=[
        OpenPoint(position="40", text="Zeichnung HB-30200 noch in Freigabe, wird bis 30.10.2026 nachgereicht; "
                                      "vorerst als Richtpreis anbieten"),
    ],
    notes=[
        "Nullserie Anfang 2027 geplant, Serie ab dem zweiten Quartal 2027",
        "Technische Fragen zu den Zeichnungen an Daniel Frei, Konstruktion",
    ],
)

In [ ]:
# Hier wird der Output vom LLM mit der Erwartung (unserem Label) überprüft.

# Listen von Objekten werden über diese Felder zugeordnet, nicht über die Reihenfolge
MATCH_KEYS = ("position", "filename", "name")
# Freitext: wird zur Sichtprüfung angezeigt, aber nicht gezählt
FREE_TEXT = {"address", "label", "process_notes", "requirements", "open_points", "notes"}


def compare(got, expected, path=""):
    """Jedes Feld der Lösung als (Pfad, extrahierter Wert, erwarteter Wert)."""
    if isinstance(expected, dict):
        for key, value in expected.items():
            sub_path = f"{path}.{key}" if path else key
            got_value = (got or {}).get(key)
            if key in FREE_TEXT:
                yield sub_path, got_value, value
            else:
                yield from compare(got_value, value, sub_path)
    elif isinstance(expected, list) and expected and isinstance(expected[0], dict):
        match_key = next((k for k in MATCH_KEYS if k in expected[0]), None)
        got = got or []
        if match_key is None:  # z. B. Mengenstaffeln: nach Reihenfolge
            for i, item in enumerate(expected):
                yield from compare(got[i] if i < len(got) else None, item, f"{path}[{i}]")
            return
        got_by_key = {item.get(match_key): item for item in got}
        for item in expected:
            yield from compare(got_by_key.get(item[match_key]), item, f"{path}[{item[match_key]}]")
        for key in got_by_key.keys() - {item[match_key] for item in expected}:  # zusätzlich extrahiert
            yield f"{path}[{key}]", got_by_key[key], None
    else:
        yield path, got, expected


def is_free_text(path: str) -> bool:
    return path.split(".")[-1].split("[")[0] in FREE_TEXT


# mode="json" macht aus Daten und Enums auf beiden Seiten Strings
results = list(compare(rfq.model_dump(mode="json"), EXPECTED.model_dump(mode="json")))
structured = [r for r in results if not is_free_text(r[0])]
wrong = [(path, got, expected) for path, got, expected in structured if got != expected]

print(f"Strukturierte Felder: {len(structured) - len(wrong)}/{len(structured)} stimmen exakt\n")
for path, got, expected in wrong:
    print(f"{path}\n  extrahiert: {got!r}\n  erwartet:   {expected!r}")

print("\nFreitext zur Sichtprüfung:")
for path, got, expected in results:
    if is_free_text(path) and got != expected:
        print(f"{path}\n  extrahiert: {got!r}\n  erwartet:   {expected!r}")

print("\nFehlende Angaben:")
got_missing, expected_missing = missing_information(rfq), missing_information(EXPECTED)
for item in dict.fromkeys(expected_missing + got_missing):
    status = "ok" if item in got_missing and item in expected_missing else (
        "nicht gefunden" if item in expected_missing else "zusätzlich")
    print(f"- [{status}] {item}")